# BrainWashed model, step 2: fine-tune, export to GGUF, evaluate

Fine-tunes a small Apache-2.0 multimodal model with LoRA on the data from step 1, merges it, converts it to GGUF (Q4_K_M and Q8_0 plus the vision projector, the format the BrainWashed host runs), and scores it against the base model on skills and tools it never saw in training.

Only the language part is trained. The vision encoder is left untouched, so the model keeps understanding images; the last cells check that it still does.

**Before you run it**
1. Settings: **Accelerator → GPU T4 x2** (training uses one GPU), **Internet → On**.
2. **Add Input → Your Work →** the step 1 notebook, so its `train.jsonl` is available.
3. To publish the model on Hugging Face automatically: add a **write** token under **Add-ons → Secrets** with the label `HF_TOKEN` and attach it to this notebook. It publishes only if the fine-tune beats the base model on the eval. Without the token the notebook still runs and skips publishing.
4. **Save Version → Save & Run All (Commit)**. Time depends on the model and data size; check the training progress bar after the first few minutes for an estimate.

In [ ]:
# ---- settings ---------------------------------------------------------------
REPO = "https://github.com/ahmadalshouly/brainwashed"
BRANCH = "main"

# Small, Apache-2.0, multimodal, tool-calling bases that llama.cpp runs. Pick one:
#   "Qwen/Qwen3.5-2B"         2B, text + images + video (default)
#   "Qwen/Qwen3.5-4B"         4B, same, noticeably smarter; set LOAD_IN_4BIT = True
#   "google/gemma-4-E2B-it"   2.3B effective (5.1B stored), text + images + audio; set LOAD_IN_4BIT = True
# A text-only model such as "Qwen/Qwen3-1.7B" also works.
BASE_MODEL = "Qwen/Qwen3.5-2B"
LOAD_IN_4BIT = False       # set True (QLoRA) for models over ~2.5B, so they fit in 16 GB
FAST_KERNELS = False       # Qwen3.5 only: try flash-linear-attention for faster training (untested on T4)
MAX_LEN = 3072             # longer conversations are dropped, not cut
EPOCHS = 2
LEARNING_RATE = 2e-4
LORA_R, LORA_ALPHA = 16, 32
BATCH_SIZE, GRAD_ACCUM = 2, 8   # 16 sequences per optimizer step

OUTPUT_NAME = "brainwashed-2b"
PUBLISH = True                  # upload to Hugging Face when an HF_TOKEN secret is attached
PUBLISH_ONLY_IF_BETTER = True   # skip the upload if the fine-tune does not beat the base model
PUBLISH_WEIGHTS = True          # also upload full-precision weights, so others can fine-tune further
HF_NAMESPACE = None             # None = your HF username; or an organization you belong to
HF_DATA_REPO = None             # None = <namespace>/brainwashed-training-data, linked from the card if it exists
HF_PRIVATE = False              # True keeps the uploaded repos private
LLAMA_CPP_REF = "master"   # pin a release tag like "b6500" for reproducible builds

In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "0"   # one GPU: simpler and steadier than splitting
os.environ["TOKENIZERS_PARALLELISM"] = "false"
# Versions this notebook was tested with. Newer transformers releases rename training options.
!pip install -q transformers==5.18.0 peft==0.21.2 accelerate==1.15.0 datasets==5.0.1 "bitsandbytes>=0.45" huggingface_hub pillow
if FAST_KERNELS:
    import subprocess
    subprocess.run("pip install -q flash-linear-attention causal-conv1d", shell=True, check=True)

In [ ]:
import os, subprocess, time, urllib.request

def sh(cmd):
    print("$", cmd, flush=True)
    subprocess.run(cmd, shell=True, check=True)

def build_llama_cpp(targets="llama-server llama-quantize"):
    """Builds llama.cpp with CUDA for this GPU (about 10 minutes the first time)."""
    if not os.path.exists("/tmp/llama.cpp"):
        sh(f"git clone --depth 1 --branch {LLAMA_CPP_REF} https://github.com/ggml-org/llama.cpp /tmp/llama.cpp"
           if LLAMA_CPP_REF != "master" else "git clone --depth 1 https://github.com/ggml-org/llama.cpp /tmp/llama.cpp")
    arch = subprocess.check_output("nvidia-smi --query-gpu=compute_cap --format=csv,noheader", shell=True, text=True).split()[0].replace(".", "")
    sh(f"cmake -S /tmp/llama.cpp -B /tmp/llama.cpp/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES={arch} -DLLAMA_CURL=OFF -DCMAKE_BUILD_TYPE=Release > /tmp/cmake.log")
    sh(f"cmake --build /tmp/llama.cpp/build --config Release -j {os.cpu_count()} --target {targets} > /tmp/build.log")

def start_llama_server(model_path, port, extra_args="", log="/tmp/llama-server.log"):
    """Starts llama-server the way the BrainWashed host does (--jinja) and waits until it is ready."""
    cmd = (f"/tmp/llama.cpp/build/bin/llama-server -m {model_path} --jinja -ngl 999 "
           f"--host 127.0.0.1 --port {port} {extra_args}")
    print("$", cmd)
    proc = subprocess.Popen(cmd, shell=True, stdout=open(log, "w"), stderr=subprocess.STDOUT)
    for _ in range(600):
        try:
            if urllib.request.urlopen(f"http://127.0.0.1:{port}/health", timeout=2).status == 200:
                print("llama-server is ready")
                return proc
        except Exception:
            pass
        if proc.poll() is not None:
            break
        time.sleep(2)
    print(open(log).read()[-3000:])
    raise RuntimeError("llama-server did not start; see the log above")


if not os.path.exists("/kaggle/working/brainwashed"):
    sh(f"git clone --depth 1 --branch {BRANCH} {REPO} /kaggle/working/brainwashed")
import sys, glob
sys.path.insert(0, "/kaggle/working/brainwashed/model")

found = glob.glob("/kaggle/input/**/train.jsonl", recursive=True) + glob.glob("/kaggle/working/data/train.jsonl")
assert found, "No train.jsonl found. Add the step 1 notebook under Add Input."
DATA_DIR = os.path.dirname(found[0])
print("data:", DATA_DIR)

## Tokenize

Each conversation becomes one example per assistant turn. The prompt is rendered with the model's own chat template, exactly as llama-server will render it, and loss is only computed on the reply.

In [ ]:
import json, numpy as np, torch
from transformers import AutoProcessor, AutoTokenizer
from bwmodel.format import render_pairs

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)
try:
    processor = AutoProcessor.from_pretrained(BASE_MODEL)
except Exception:
    processor = None   # text-only model
if tokenizer.chat_template is None and processor is not None:
    tokenizer.chat_template = processor.chat_template
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def apply(messages, tools=None, add_generation_prompt=False):
    return tokenizer.apply_chat_template(messages, tools=tools, add_generation_prompt=add_generation_prompt, tokenize=False)

def load(name):
    examples, skipped, too_long = [], 0, 0
    for line in open(f"{DATA_DIR}/{name}"):
        pairs, bad = render_pairs(json.loads(line), apply)
        skipped += bad
        for prompt, completion in pairs:
            p = tokenizer(prompt, add_special_tokens=False)["input_ids"]
            c = tokenizer(completion, add_special_tokens=False)["input_ids"]
            if len(p) + len(c) > MAX_LEN:
                too_long += 1
                continue
            examples.append({"input_ids": p + c, "labels": [-100] * len(p) + c})
    lengths = [len(e["input_ids"]) for e in examples]
    print(f"{name}: {len(examples)} examples, {skipped} turns did not render cleanly, {too_long} too long; "
          f"tokens median {int(np.median(lengths))}, 95th pct {int(np.percentile(lengths, 95))}")
    return examples

train_examples, val_examples = load("train.jsonl"), load("val.jsonl")

# Show one example exactly as the model sees it, with the trained part marked.
e = train_examples[0]
start = next(i for i, l in enumerate(e["labels"]) if l != -100)
print(tokenizer.decode(e["input_ids"][:start])[-1500:] + "\n>>>>>> TRAINED ON >>>>>>\n" + tokenizer.decode(e["input_ids"][start:]))

## Train

In [ ]:
from datasets import Dataset
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (AutoModelForCausalLM, AutoModelForImageTextToText, BitsAndBytesConfig,
                          DataCollatorForSeq2Seq, Trainer, TrainingArguments)

def load_model(**kwargs):
    """Loads the full model, vision (and audio) encoders included, so they survive the merge."""
    try:
        return AutoModelForImageTextToText.from_pretrained(BASE_MODEL, **kwargs), True
    except ValueError:
        return AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kwargs), False

bf16 = torch.cuda.is_bf16_supported()
if LOAD_IN_4BIT:
    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                               bnb_4bit_compute_dtype=torch.bfloat16 if bf16 else torch.float16)
    model, MULTIMODAL = load_model(quantization_config=quant, device_map={"": 0})
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
else:
    # T4s have no bfloat16, and some models overflow in pure float16, so keep
    # float32 weights and let mixed precision run the math in float16.
    model, MULTIMODAL = load_model(dtype=torch.bfloat16 if bf16 else torch.float32, device_map={"": 0})
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
model.config.use_cache = False

# Attention and MLP projections of the language model only. Qwen3.5's linear-attention
# layers name theirs in_proj_qkv / in_proj_z / out_proj. Vision and audio encoders stay frozen.
PROJECTIONS = "q_proj|k_proj|v_proj|o_proj|gate_proj|up_proj|down_proj|in_proj_qkv|in_proj_z|out_proj"
target = rf".*language_model.*\.({PROJECTIONS})" if MULTIMODAL else rf".*\.({PROJECTIONS})"
model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=0.05, task_type="CAUSAL_LM", target_modules=target))
model.print_trainable_parameters()
print("multimodal:", MULTIMODAL)

args = TrainingArguments(
    output_dir="/kaggle/working/checkpoints",
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_steps=max(1, int(0.03 * EPOCHS * len(train_examples) / (BATCH_SIZE * GRAD_ACCUM))),
    weight_decay=0.0,
    fp16=not bf16,
    bf16=bf16,
    logging_steps=10,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,
    report_to="none",
    remove_unused_columns=False,
)
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=Dataset.from_list(train_examples),
    eval_dataset=Dataset.from_list(val_examples),
    data_collator=DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100),
)
resume = bool(glob.glob("/kaggle/working/checkpoints/checkpoint-*"))
trainer.train(resume_from_checkpoint=resume or None)
model.save_pretrained("/kaggle/working/lora-adapter")
tokenizer.save_pretrained("/kaggle/working/lora-adapter")

## Merge the adapter into the base model

The adapter is merged into a fresh float16 copy of the base model (merging into 4-bit weights would lose quality).

In [ ]:
import gc
from peft import PeftModel
del model, trainer
gc.collect(); torch.cuda.empty_cache()

base, _ = load_model(dtype=torch.float16, device_map={"": "cpu"})
merged = PeftModel.from_pretrained(base, "/kaggle/working/lora-adapter").merge_and_unload()
merged.save_pretrained("/tmp/merged", safe_serialization=True)
tokenizer.save_pretrained("/tmp/merged")
if processor is not None:
    processor.save_pretrained("/tmp/merged")
del base, merged
gc.collect()

## Convert to GGUF

Builds llama.cpp, converts the merged model and the untouched base model (for the comparison below), and quantizes. For a multimodal model it also writes the vision projector (`mmproj-*.gguf`), which llama-server loads with `--mmproj`. The files you ship land in `/kaggle/working/gguf`.

In [ ]:
import re
from huggingface_hub import snapshot_download

build_llama_cpp("llama-server llama-quantize")

def convert(src, out, extra=""):
    """Runs llama.cpp's converter, installing any small module it turns out to need."""
    for _ in range(5):
        r = subprocess.run(f"python /tmp/llama.cpp/convert_hf_to_gguf.py {src} --outtype f16 --outfile {out} {extra}",
                           shell=True, capture_output=True, text=True)
        missing = re.search(r"No module named '([\w\-]+)", r.stderr)
        if r.returncode == 0:
            return
        if not missing:
            print(r.stderr[-3000:]); raise RuntimeError("conversion failed")
        sh(f"pip install -q {missing.group(1).replace('_', '-')}")
    raise RuntimeError("conversion failed")

os.makedirs("/kaggle/working/gguf", exist_ok=True)
convert("/tmp/merged", "/tmp/finetuned-f16.gguf")
for quant in ["Q4_K_M", "Q8_0"]:
    sh(f"/tmp/llama.cpp/build/bin/llama-quantize /tmp/finetuned-f16.gguf /kaggle/working/gguf/{OUTPUT_NAME}-{quant}.gguf {quant} > /dev/null")
MMPROJ = f"/kaggle/working/gguf/mmproj-{OUTPUT_NAME}-f16.gguf"
if MULTIMODAL:
    convert("/tmp/merged", MMPROJ, "--mmproj")

convert(snapshot_download(BASE_MODEL, local_dir="/tmp/base-hf"), "/tmp/base-f16.gguf")
sh("/tmp/llama.cpp/build/bin/llama-quantize /tmp/base-f16.gguf /tmp/base-Q4_K_M.gguf Q4_K_M > /dev/null")
!ls -lh /kaggle/working/gguf

## Evaluate on unseen skills and tools

Both models run as Q4_K_M GGUF through llama-server with `--jinja`, which is exactly how the host runs them. The base model gets its best non-thinking setting; the fine-tuned model runs with no extra settings, like in the app.

In [ ]:
EVAL = "cd /kaggle/working/brainwashed && python -u model/eval/run_eval.py --base-url http://127.0.0.1:8081/v1"
NO_THINK = """'{"chat_template_kwargs": {"enable_thinking": false}}'"""
reports = {}
for label, path, extra in [("base", "/tmp/base-Q4_K_M.gguf", f"--extra-body {NO_THINK}"),
                           ("fine-tuned", f"/kaggle/working/gguf/{OUTPUT_NAME}-Q4_K_M.gguf", "")]:
    mmproj = f"--mmproj {MMPROJ}" if MULTIMODAL and label == "fine-tuned" else ""
    server = start_llama_server(path, 8081, f"-c 8192 -np 1 {mmproj}")
    try:
        sh(f"{EVAL} --label {label} --out /kaggle/working/eval-{label}.json {extra}")
    finally:
        server.terminate(); server.wait()
    reports[label] = json.load(open(f"/kaggle/working/eval-{label}.json"))["summary"]

cats = sorted(set(reports["base"]["by_category"]) | set(reports["fine-tuned"]["by_category"]))
print(f"\n{'':<22}{'base':>8}{'fine-tuned':>12}")
print(f"{'ALL':<22}{reports['base']['pass_rate']:>8.0%}{reports['fine-tuned']['pass_rate']:>12.0%}")
for c in cats:
    print(f"{c:<22}{reports['base']['by_category'].get(c, 0):>8.0%}{reports['fine-tuned']['by_category'].get(c, 0):>12.0%}")

## Check that it still sees images

Draws a simple picture and asks the fine-tuned model about it through llama-server, the same way the host would send a photo from the phone.

In [ ]:
if MULTIMODAL:
    import base64, io
    from PIL import Image, ImageDraw
    img = Image.new("RGB", (448, 448), "white")
    ImageDraw.Draw(img).ellipse((80, 80, 368, 368), fill="red")
    buf = io.BytesIO(); img.save(buf, format="PNG")
    body = {"messages": [{"role": "user", "content": [
        {"type": "image_url", "image_url": {"url": "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()}},
        {"type": "text", "text": "What shape is in this picture, and what color is it? Answer in a few words."}]}],
        "temperature": 0, "max_tokens": 60}
    server = start_llama_server(f"/kaggle/working/gguf/{OUTPUT_NAME}-Q4_K_M.gguf", 8081, f"-c 8192 -np 1 --mmproj {MMPROJ}")
    try:
        req = urllib.request.Request("http://127.0.0.1:8081/v1/chat/completions", data=json.dumps(body).encode(),
                                     headers={"Content-Type": "application/json"})
        answer = json.loads(urllib.request.urlopen(req, timeout=300).read())["choices"][0]["message"]["content"]
        print("model says:", answer)
        print("vision OK" if "red" in answer.lower() and "circle" in answer.lower() else "check the answer above")
    finally:
        server.terminate(); server.wait()

## Publish to Hugging Face

Uploads the GGUF files to `<you>/<OUTPUT_NAME>-GGUF` and the full-precision weights to `<you>/<OUTPUT_NAME>`, each with a model card that includes the eval table. Skipped when there is no `HF_TOKEN` secret, or when the fine-tune does not beat the base model.

In [ ]:
from bwmodel.publish import gguf_card, hf_token, should_publish, weights_card

token = hf_token()
ok, reason = should_publish(reports, PUBLISH_ONLY_IF_BETTER)
if not PUBLISH:
    print("not publishing: PUBLISH is False")
elif not token:
    print("not publishing: no HF_TOKEN secret attached (Add-ons -> Secrets)")
elif not ok:
    print("not publishing:", reason)
else:
    from huggingface_hub import HfApi
    api = HfApi(token=token)
    namespace = HF_NAMESPACE or api.whoami()["name"]
    gguf_repo, weights_repo = f"{namespace}/{OUTPUT_NAME}-GGUF", f"{namespace}/{OUTPUT_NAME}"
    data_repo = HF_DATA_REPO or f"{namespace}/brainwashed-training-data"
    if not api.repo_exists(data_repo, repo_type="dataset"):
        data_repo = None
    print(reason)

    open("/kaggle/working/gguf/README.md", "w").write(gguf_card(
        OUTPUT_NAME, BASE_MODEL, reports, weights_repo if PUBLISH_WEIGHTS else None, data_repo, MULTIMODAL))
    api.create_repo(gguf_repo, private=HF_PRIVATE, exist_ok=True)
    api.upload_folder(repo_id=gguf_repo, folder_path="/kaggle/working/gguf", commit_message="Upload GGUF files")
    print(f"published: https://huggingface.co/{gguf_repo}")

    if PUBLISH_WEIGHTS:
        open("/tmp/merged/README.md", "w").write(weights_card(OUTPUT_NAME, BASE_MODEL, reports, gguf_repo, data_repo))
        api.create_repo(weights_repo, private=HF_PRIVATE, exist_ok=True)
        api.upload_folder(repo_id=weights_repo, folder_path="/tmp/merged", commit_message="Upload merged weights")
        print(f"published: https://huggingface.co/{weights_repo}")